This notebook shows how to use the general interface in HiGP for regression problems.

In this test, we will use the 3D Road dataset at "https://archive.ics.uci.edu/ml/machine-learning-databases/00246/3D_spatial_network.txt". 

We random sample 30000 points for training and use 100 points for testing for demonstration.

In [3]:
import higp
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import torch
%matplotlib inline

Since this is a large (> 20000 training points) and low-dimensional (2D or 3D) data set, HiGP can use the $\mathcal{H}^2$ matrix for faster calculation. The $\mathcal{H}^2$ matrix requires a higher working precision, so we use float64 instead of float32 in this example.

In [4]:
np_dtype = np.float64

Download the dataset, Scale features to $[-1, 1]$ and normalize labels.

In [5]:
df = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/00246/3D_spatial_network.txt", sep=',', header=None)
data_array = df.values[:, 1:]
all_x = data_array[:, 1:3]
all_y = data_array[:, -1]

# Scale features and normalize labels
all_x_max = np.max(all_x, 0)
all_x_min = np.min(all_x, 0)
all_x = 2.0 * (all_x - all_x_min[np.newaxis, :]) / (all_x_max[np.newaxis, :] - all_x_min[np.newaxis, :]) - 1.0
all_y = (all_y - np.mean(all_y)) / np.std(all_y)

# Randomly select a subset of the data
n_all = all_x.shape[0]
n_train = 30000
n_test = 100
n_sample = n_train + n_test

sample_array = np.random.choice(n_all, n_sample, replace = False)

train_x = np.ascontiguousarray(all_x[sample_array[:n_train], :].T).astype(np_dtype)
train_y = np.ascontiguousarray(all_y[sample_array[:n_train]]).astype(np_dtype)
test_x = np.ascontiguousarray(all_x[sample_array[n_train:], :].T).astype(np_dtype)
test_y = np.ascontiguousarray(all_y[sample_array[n_train:]]).astype(np_dtype)

Remember to use the `ascontiguousarray()` method in NumPy to guarantee that `train_x, train_y, test_x, test_y` are stored contiguously.

Now let's check the shapes of these four arrays. We can see that each data point is stored in one column in `train_x` and `test_x`.

In [6]:
print("Shape of train_x : ", train_x.shape)
print("Shape of train_y : ", train_y.shape)
print("Shape of test_x : ", test_x.shape)
print("Shape of test_y : ", test_y.shape)

Shape of train_x :  (2, 30000)
Shape of train_y :  (30000,)
Shape of test_x :  (2, 100)
Shape of test_y :  (100,)


Create a GP regression problem model and a PyTorch Adam optimizer. 

By default, HiGP uses the $\mathcal{H}^2$ matrix if possible (`mvtype = higp.MatvecAuto`). If you want to disable the use of the $\mathcal{H}^2$ matrix, set `mvtype = higp.MatvecAOT`. For more information about the parameter `mvtype`, please refer to the user manual. 

In [7]:
torch_dtype = torch.float32 if np_dtype == np.float32 else torch.float64
gprproblem = higp.gprproblem.setup(data=train_x, label=train_y, kernel_type=higp.GaussianKernel, mvtype=higp.MatvecAuto)
model = higp.GPRModel(gprproblem, dtype=torch_dtype)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

Setting up GPR problem with 30000 samples and 2 features
Init gpr problem done
Creating GPR problem
Data type: double
Kernel type: Gaussian
Transform type: softplus
Using preconditioned GP, kernel matrix form: H2 matrix
AFN preconditioner parameters: rank 50, lfil 0
LanQ parameters: niter 10, nvec 10


Run 20 steps of Adam.

In [8]:
loss_history, param_histpry = higp.gpr_torch_minimize(model, optimizer, maxits=20, print_info=True)

Iteration (max 20), Elapsed time (sec), Loss, Hyperparameters (l, s, f, before nnt)


/home/chab/Hmat dossier code/hmat_env/lib/python3.14/site-packages/torch/cuda/__init__.py:228: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


1, 17.44, 0.73839, 0.050, -0.050, 0.050
2, 38.03, 0.72015, 0.100, -0.091, 0.100
3, 56.62, 0.70501, 0.149, -0.136, 0.150
4, 73.20, 0.68830, 0.198, -0.183, 0.199
5, 89.79, 0.67089, 0.246, -0.230, 0.248
6, 106.48, 0.65288, 0.292, -0.277, 0.297
7, 123.11, 0.63506, 0.338, -0.315, 0.345
8, 140.50, 0.62027, 0.383, -0.357, 0.393
9, 156.90, 0.60401, 0.426, -0.402, 0.441
10, 173.51, 0.58666, 0.467, -0.442, 0.488
11, 190.43, 0.57067, 0.507, -0.479, 0.534
12, 206.29, 0.55614, 0.545, -0.519, 0.580
13, 223.78, 0.54030, 0.581, -0.559, 0.625
14, 240.60, 0.52420, 0.616, -0.600, 0.669
15, 256.58, 0.50732, 0.650, -0.644, 0.713
16, 273.35, 0.48969, 0.681, -0.688, 0.756
17, 289.72, 0.47152, 0.712, -0.731, 0.799
18, 308.24, 0.45363, 0.740, -0.776, 0.840
19, 325.13, 0.43518, 0.768, -0.822, 0.881
20, 342.45, 0.41591, 0.795, -0.868, 0.922


Run predictions with the initial parameters and the trained parameters.

In [9]:
Pred0 = higp.gpr_prediction(data_train=train_x,
                            label_train=train_y,
                            data_prediction=test_x,
                            kernel_type=higp.GaussianKernel,
                            gp_params=np.hstack((0.0, 0.0, 0.0)))

Pred = higp.gpr_prediction(data_train=train_x,
                           label_train=train_y,
                           data_prediction=test_x,
                           kernel_type=higp.GaussianKernel,
                           gp_params=model.get_params())

Running GPR prediction
Data type: double
Kernel type: Gaussian
Transform type: softplus
Using preconditioned GP, kernel matrix form: H2 matrix
AFN preconditioner parameters: rank 50, lfil 0
PCG parameters: niter 50, tol 1e-06
Running GPR prediction
Data type: double
Kernel type: Gaussian
Transform type: softplus
Using preconditioned GP, kernel matrix form: H2 matrix
AFN preconditioner parameters: rank 50, lfil 0
PCG parameters: niter 50, tol 1e-06


Finally, let us check the root mean squared error (RMSE) of the predition.

In [10]:
rmse0 = np.linalg.norm(Pred0.prediction_mean - test_y) / np.sqrt(float(n_test))
rmse = np.linalg.norm(Pred.prediction_mean - test_y) / np.sqrt(float(n_test))
print("RMSE (before training): %g, RMSE (after training): %g\n" % (rmse0, rmse))

RMSE (before training): 0.00899811, RMSE (after training): 0.00407281

